# Claude Code — Hooks

---

## What Hooks Are

Run a command **before or after** Claude Code executes a tool.
Optionally **block** Claude's action entirely.
```
Your request
      |
      v
PreToolUse hook   ← can block the tool call + send error to Claude
      |
      v
Claude runs the tool  (e.g. reads a file, edits a file)
      |
      v
PostToolUse hook  ← too late to block, but can send feedback to Claude
```

---

## Two Hook Types

| Hook | When it runs | Can block? | Can send feedback? |
|---|---|---|---|
| `PreToolUse` | Before the tool executes | Yes | Yes (error message) |
| `PostToolUse` | After the tool executes | No | Yes (additional context) |

---

## Hook Configuration Locations

| File | Scope |
|---|---|
| `~/.claude/settings.json` | All projects on your machine |
| `.claude/settings.json` | Project-wide, committed to source control |
| `.claude/settings.local.json` | Project-local, not committed |

Configure by hand in these files or use the `/hooks` command inside Claude Code.

---

## Configuration Structure

### PreToolUse — runs before the tool, can block
```json
"PreToolUse": [
  {
    "matcher": "Read",
    "hooks": [
      {
        "type": "command",
        "command": "node /home/hooks/read_hook.ts"
      }
    ]
  }
]
```

Before any `Read` tool call, the hook script runs. The script receives details
about what Claude wants to read and can allow or block the operation.

### PostToolUse — runs after the tool, can give feedback
```json
"PostToolUse": [
  {
    "matcher": "Write|Edit|MultiEdit",
    "hooks": [
      {
        "type": "command",
        "command": "node /home/hooks/edit_hook.ts"
      }
    ]
  }
]
```

After any write or edit operation, the hook runs. Can trigger follow-up actions
like formatting the file that was just changed.

---

## Practical Use Cases

| Use Case | Hook type |
|---|---|
| Run code formatter after Claude edits a file | `PostToolUse` on Write/Edit |
| Run tests automatically when a file changes | `PostToolUse` on Write/Edit |
| Block Claude from reading a specific file | `PreToolUse` on Read |
| Block edits that violate naming conventions | `PreToolUse` on Write/Edit |
| Check for TODO comments in code Claude writes | `PostToolUse` on Write |
| Block deprecated function usage | `PreToolUse` on Write/Edit |
| Log every file Claude accesses or modifies | Either hook type |

---

## Key Takeaway

> `PreToolUse` hooks give you **control** — decide what Claude is allowed to do.  
> `PostToolUse` hooks give you **enhancement** — react to what Claude has done.  
> Together they let you integrate your own tools, standards, and processes
> directly into Claude Code's workflow.

# Claude Code — Building Hooks

---

## Four Steps to Build a Hook
```
1. Choose hook type      → PreToolUse (can block) or PostToolUse (feedback only)
2. Choose which tool     → Read, Write, Edit, Bash, Grep, etc.
3. Write a command       → receives JSON via stdin about the tool call
4. Return feedback       → exit code tells Claude allow or block
```

---

## Tool Call Data — What Your Hook Receives

When your hook runs, Claude sends JSON to **stdin**:
```json
{
  "session_id": "2d6a1e4d-6...",
  "transcript_path": "/Users/sg/...",
  "hook_event_name": "PreToolUse",
  "tool_name": "Read",
  "tool_input": {
    "file_path": "/code/queries/.env"
  }
}
```

Your script reads this from stdin, parses it, checks the `tool_name` and
`tool_input`, then decides what to do.

---

## Exit Codes — How Your Hook Communicates Back

| Exit code | Meaning |
|---|---|
| `0` | Allow the tool call to proceed normally |
| `2` | Block the tool call (PreToolUse only) — any stderr output is sent to Claude as the reason |

---

## Example — Block Reads of `.env` Files
```typescript
// read_hook.ts
import * as readline from "readline";

const rl = readline.createInterface({ input: process.stdin });
let input = "";

rl.on("line", (line) => (input += line));
rl.on("close", () => {
  const data = JSON.parse(input);
  const filePath = data.tool_input?.file_path ?? "";

  if (filePath.endsWith(".env")) {
    process.stderr.write("Blocked: Claude is not allowed to read .env files.");
    process.exit(2);   // block the call
  }

  process.exit(0);     // allow everything else
});
```

Register it in `.claude/settings.json`:
```json
"PreToolUse": [
  {
    "matcher": "Read|Grep",
    "hooks": [
      {
        "type": "command",
        "command": "node /home/hooks/read_hook.ts"
      }
    ]
  }
]
```

Both `Read` and `Grep` can access file contents — monitor both to fully protect
sensitive files.

---

## Available Tools to Monitor

Ask Claude directly for the current list:
```
What tools do you have available?
```

The list can change when MCP servers are added.

Common built-in tools: `Read`, `Write`, `Edit`, `MultiEdit`, `Bash`, `Grep`,
`Glob`, `LS`, `WebFetch`, `NotebookRead`, `NotebookEdit`, `TodoRead`, `TodoWrite`

---

## Hook Decision Flow
```
Claude wants to Read /code/.env
         |
         v
PreToolUse hook runs read_hook.ts
         |
         v
Script parses JSON from stdin
         |
    .env detected?
    Yes → stderr: "Blocked..."
          exit(2) → Claude receives error message, does not read the file
    No  → exit(0) → Claude reads the file normally
```

---

## Key Takeaway

> Your hook is just a script that reads JSON from stdin and exits with 0 or 2.  
> Exit 0 = allow. Exit 2 = block + tell Claude why via stderr.  
> Monitor multiple tool types (e.g. Read AND Grep) when the same data
> can be accessed through different tools.

# Claude Code — Hook Implementation (`.env` Protection)

---

## Settings Configuration

In `.claude/settings.local.json`:
```json
{
  "hooks": {
    "PreToolUse": [
      {
        "matcher": "Read|Grep",
        "hooks": [
          {
            "type": "command",
            "command": "node ./hooks/read_hook.js"
          }
        ]
      }
    ]
  }
}
```

`|` acts as OR — hook triggers on either `Read` or `Grep`.

---

## Hook Script — `./hooks/read_hook.js`
```javascript
async function main() {
  const chunks = [];
  for await (const chunk of process.stdin) {
    chunks.push(chunk);
  }

  const toolArgs = JSON.parse(Buffer.concat(chunks).toString());

  // Handle both file_path and path field names
  const readPath =
    toolArgs.tool_input?.file_path || toolArgs.tool_input?.path || "";

  if (readPath.includes(".env")) {
    console.error("You cannot read the .env file");
    process.exit(2);   // block + send error message to Claude
  }
}

main();
```

---

## After Setup

1. Save both files
2. Restart Claude Code
3. Ask Claude to read `.env` → hook intercepts → Claude receives the error
   and explains the operation was blocked

---

## Key Takeaway

> The pattern is always the same: read stdin → parse JSON →
> check `tool_input` → exit 2 to block, exit 0 to allow.  
> Extend `readPath.includes()` with additional patterns to protect
> more files or directories.

# Claude Code — Hook Security & Sharing Settings Files

---

## Security Recommendation

Anthropic recommends using **absolute paths** for hook scripts — not relative paths.

**Why:** Prevents path interception and binary planting attacks where a malicious
script could be placed at a relative path and executed instead of your intended script.

---

## The Problem This Creates

Absolute paths are machine-specific:
```
Your machine:  /Users/yourname/projects/uigen/hooks/read_hook.js
My machine:    /Users/myname/dev/uigen/hooks/read_hook.js
```

A `settings.json` with hardcoded absolute paths **cannot be shared** via source control.

---

## The Solution — `$PWD` Placeholder + Init Script

The project uses a `settings.example.json` with a placeholder:
```json
{
  "hooks": {
    "PreToolUse": [
      {
        "matcher": "Read|Grep",
        "hooks": [
          {
            "type": "command",
            "command": "node $PWD/hooks/read_hook.js"
          }
        ]
      }
    ]
  }
}
```

When you run `npm run setup`, an `init-claude.js` script:
```
1. Reads settings.example.json
2. Replaces $PWD with the actual absolute path of your machine
3. Writes the result to settings.local.json  ← your personal, non-committed file
```

---

## Why Two Settings Files in `.claude/`?

| File | Purpose |
|---|---|
| `settings.json` | Shared project settings — committed to source control |
| `settings.local.json` | Machine-specific generated file — gitignored, contains absolute paths |

`settings.local.json` is regenerated per machine via `npm run setup` — never manually edited.

---

## Key Takeaway

> Use absolute paths in hook commands for security.  
> Use a `$PWD` placeholder in a shared example file and replace it at setup time.  
> This lets you share hook configurations across a team without hardcoding
> machine-specific paths.

# Claude Code — Practical Hook Examples

---

## Hook 1 — TypeScript Type Checking

**Problem:** Claude updates a function signature but misses all the call sites
in other files, leaving type errors Claude never sees.

**Solution:** `PostToolUse` hook that runs `tsc` after every file edit and
feeds errors back to Claude immediately.
```
Claude edits a file
        |
        v
PostToolUse hook runs: tsc --noEmit
        |
        v
Errors found? → send error output back to Claude
        |
Claude fixes the call sites in other files
```

**Hook script logic:**
```javascript
// Run type checker
const result = execSync("tsc --noEmit", { capture: true });

if (result.errors) {
  // Write errors to stdout — Claude receives them as feedback
  console.log(result.errors);
}
```

Works for any typed language — for untyped languages, substitute with
automated test output instead.

---

## Hook 2 — Query Duplication Prevention

**Problem:** On large projects with many query files, Claude creates new
database queries instead of reusing existing ones.

**Example:** You ask "create a Slack integration that alerts about orders
pending longer than 3 days" — Claude writes a new query instead of using
the existing `getPendingOrders()` function.

**Solution:** `PostToolUse` hook on the `./queries` directory that launches
a second Claude instance to review for duplicates.
```
Claude edits a file in ./queries
        |
        v
PostToolUse hook triggers
        |
        v
Hook launches a second Claude instance programmatically
        |
        v
Second Claude: "Does this new query duplicate any existing functionality?"
        |
   Duplicate found?
   Yes → send feedback to original Claude: "Use getPendingOrders() instead"
   No  → allow, no feedback needed
```

---

## Trade-offs

| Hook | Cost | Benefit |
|---|---|---|
| TypeScript checker | Low — fast compile | Catches type errors across files immediately |
| Query duplicate review | Higher — launches another Claude instance | Keeps codebase clean, prevents redundant queries |

**Recommendation for query hook:** Only monitor high-value, critical directories
to keep overhead manageable.

---

## Broader Principles These Demonstrate

| Principle | Example |
|---|---|
| Use compiler/linter output as feedback | `tsc --noEmit` errors fed to Claude |
| AI reviewing AI | Second Claude instance checks first Claude's work |
| Target high-value directories only | Watch `./queries`, not the whole codebase |
| Balance automation vs cost | Lightweight hooks everywhere, heavy hooks sparingly |

---

## Key Takeaway

> Hooks turn Claude Code from a one-shot assistant into a self-correcting
> development loop.  
> Feed compiler errors back automatically so Claude fixes what it broke.  
> Use a second Claude instance to review changes in critical areas where
> consistency matters most.

# Claude Code — Full Hook Reference

---

## All Hook Types

| Hook | When it runs |
|---|---|
| `PreToolUse` | Before a tool is called — can block |
| `PostToolUse` | After a tool is called — can give feedback |
| `Notification` | When Claude needs permission for a tool, or after 60s idle |
| `Stop` | When Claude has finished responding |
| `SubagentStop` | When a subagent (shown as "Task" in UI) finishes |
| `PreCompact` | Before a compact operation (manual or automatic) |
| `UserPromptSubmit` | When user submits a prompt, before Claude processes it |
| `SessionStart` | When starting or resuming a session |
| `SessionEnd` | When a session ends |

---

## stdin Input Varies by Hook Type

The JSON your script receives on stdin changes depending on the hook type
**and** the tool that was matched.

### `PostToolUse` on `TodoWrite`
```json
{
  "session_id": "9ecf22fa-...",
  "transcript_path": "<path>",
  "hook_event_name": "PostToolUse",
  "tool_name": "TodoWrite",
  "tool_input": {
    "todos": [{ "content": "write a readme", "status": "pending", "priority": "medium", "id": "1" }]
  },
  "tool_response": {
    "oldTodos": [],
    "newTodos": [{ "content": "write a readme", "status": "pending", "priority": "medium", "id": "1" }]
  }
}
```

### `Stop` hook
```json
{
  "session_id": "af9f50b6-...",
  "transcript_path": "<path>",
  "hook_event_name": "Stop",
  "stop_hook_active": false
}
```

Different hook type → completely different fields.

---

## The Problem

When writing a hook you may not know the exact structure of the stdin input
your script will receive — especially for unfamiliar tools or hook types.

---

## The Solution — Logging Helper Hook

Add a wildcard hook that writes stdin to a file so you can inspect it:
```json
"PostToolUse": [
  {
    "matcher": "*",
    "hooks": [
      {
        "type": "command",
        "command": "jq . > post-log.json"
      }
    ]
  }
]
```

- `matcher: "*"` matches every tool call
- `jq .` pretty-prints the JSON from stdin
- Output saved to `post-log.json` for inspection

**Workflow:**
```
1. Add the logging hook
2. Trigger the tool you want to hook into
3. Open post-log.json
4. See the exact JSON structure your real hook will receive
5. Remove the logging hook, write your real hook with confidence
```

Use the same pattern for any hook type — just change `"PostToolUse"` to
`"PreToolUse"`, `"Stop"`, `"SessionStart"`, etc.

---

## Key Takeaway

> stdin structure varies by hook type and matched tool — don't guess it.  
> Use the `jq . > log.json` helper hook to capture real input,
> then build your hook script against the actual data structure.

# Claude Code — Full Hook Reference

---

## All Hook Types

| Hook | When it runs |
|---|---|
| `PreToolUse` | Before a tool is called — can block |
| `PostToolUse` | After a tool is called — can give feedback |
| `Notification` | When Claude needs permission for a tool, or after 60s idle |
| `Stop` | When Claude has finished responding |
| `SubagentStop` | When a subagent (shown as "Task" in UI) finishes |
| `PreCompact` | Before a compact operation (manual or automatic) |
| `UserPromptSubmit` | When user submits a prompt, before Claude processes it |
| `SessionStart` | When starting or resuming a session |
| `SessionEnd` | When a session ends |

---

## stdin Input Varies by Hook Type

The JSON your script receives on stdin changes depending on the hook type
**and** the tool that was matched.

### `PostToolUse` on `TodoWrite`
```json
{
  "session_id": "9ecf22fa-...",
  "transcript_path": "<path>",
  "hook_event_name": "PostToolUse",
  "tool_name": "TodoWrite",
  "tool_input": {
    "todos": [{ "content": "write a readme", "status": "pending", "priority": "medium", "id": "1" }]
  },
  "tool_response": {
    "oldTodos": [],
    "newTodos": [{ "content": "write a readme", "status": "pending", "priority": "medium", "id": "1" }]
  }
}
```

### `Stop` hook
```json
{
  "session_id": "af9f50b6-...",
  "transcript_path": "<path>",
  "hook_event_name": "Stop",
  "stop_hook_active": false
}
```

Different hook type → completely different fields.

---

## The Problem

When writing a hook you may not know the exact structure of the stdin input
your script will receive — especially for unfamiliar tools or hook types.

---

## The Solution — Logging Helper Hook

Add a wildcard hook that writes stdin to a file so you can inspect it:
```json
"PostToolUse": [
  {
    "matcher": "*",
    "hooks": [
      {
        "type": "command",
        "command": "jq . > post-log.json"
      }
    ]
  }
]
```

- `matcher: "*"` matches every tool call
- `jq .` pretty-prints the JSON from stdin
- Output saved to `post-log.json` for inspection

**Workflow:**
```
1. Add the logging hook
2. Trigger the tool you want to hook into
3. Open post-log.json
4. See the exact JSON structure your real hook will receive
5. Remove the logging hook, write your real hook with confidence
```

Use the same pattern for any hook type — just change `"PostToolUse"` to
`"PreToolUse"`, `"Stop"`, `"SessionStart"`, etc.

---

## Key Takeaway

> stdin structure varies by hook type and matched tool — don't guess it.  
> Use the `jq . > log.json` helper hook to capture real input,
> then build your hook script against the actual data structure.

# Claude Code SDK

---

## What It Is

Run Claude Code programmatically from your own applications and scripts.
Available for **TypeScript**, **Python**, and **CLI**.

Same Claude Code as the terminal — same tools, same settings, fully automatable.

---

## Key Properties

| Property | Detail |
|---|---|
| Same tools as terminal | All built-in Claude Code tools available |
| Inherits settings | Uses `.claude/` settings from the project directory |
| Default permissions | **Read-only** — can read, search, grep; cannot write or edit |
| Best used as | Part of larger pipelines, automation scripts, CI/CD |

---

## Basic Usage (TypeScript)
```typescript
import { query } from "@anthropic-ai/claude-code";

const prompt = "Look for duplicate queries in the ./src/queries dir";

for await (const message of query({ prompt })) {
  console.log(JSON.stringify(message, null, 2));
}
```

Streams the full conversation message by message. The final message contains
Claude's complete response.

---

## Enabling Write Permissions

Default is read-only. Add `allowedTools` to enable specific write operations:
```typescript
for await (const message of query({
  prompt,
  options: {
    allowedTools: ["Edit"]
  }
})) {
  console.log(JSON.stringify(message, null, 2));
}
```

Or configure project-wide permissions in `.claude/settings.json`.

---

## Practical Use Cases

| Use case | How SDK helps |
|---|---|
| Git hooks | Auto-review code changes before commit |
| Build scripts | Analyse and optimise code as part of the build |
| CI/CD pipelines | Code quality checks on every push |
| Documentation | Auto-generate docs from code changes |
| Code maintenance | Scheduled cleanup, refactoring, duplicate detection |

---

## Key Takeaway

> The Claude Code SDK brings the full power of Claude Code into your
> own scripts and pipelines.  
> Default read-only permissions make it safe to use in automated contexts.  
> Add `allowedTools` only when write access is genuinely needed.


# The Python SDK works almost identically to TypeScript — same concepts, just Python syntax.

---

## Step 1 — Install it

```bash
pip install claude-code-sdk
```

You also need Claude Code itself installed:

```bash
npm install -g @anthropic-ai/claude-code
```

---

## Step 2 — Simplest possible usage

```python
import asyncio
from claude_code_sdk import query, ClaudeCodeOptions

async def main():
    async for message in query(
        prompt="What files are in the current directory?"
    ):
        print(message)

asyncio.run(main())
```

Python's version uses `async for` instead of TypeScript's `for await`. Same idea — processes messages as they stream in.

---

## Step 3 — Getting just the final answer

```python
import asyncio
from claude_code_sdk import query

async def main():
    final_response = ""

    async for message in query(
        prompt="Summarize what main.py does"
    ):
        if message.type == "result":
            final_response = message.result

    print(final_response)

asyncio.run(main())
```

Same pattern as TypeScript — check `message.type == "result"` to get only the completed answer.

---

## Step 4 — Allowing write permissions

```python
import asyncio
from claude_code_sdk import query, ClaudeCodeOptions

async def main():
    async for message in query(
        prompt="Fix any type errors in main.py",
        options=ClaudeCodeOptions(
            allowed_tools=["Read", "Edit", "Write"]
        )
    ):
        if message.type == "result":
            print(message.result)

asyncio.run(main())
```

Notice Python uses `ClaudeCodeOptions` as a proper object, and `allowed_tools` is snake_case instead of camelCase.

---

## Step 5 — Setting a working directory

```python
import asyncio
from claude_code_sdk import query, ClaudeCodeOptions

async def main():
    async for message in query(
        prompt="Check for duplicate functions",
        options=ClaudeCodeOptions(
            cwd="/path/to/your/project"
        )
    ):
        if message.type == "result":
            print(message.result)

asyncio.run(main())
```

---

## Step 6 — A real practical example

A code review script you could run before committing:

```python
import asyncio
import subprocess
from claude_code_sdk import query, ClaudeCodeOptions

async def review_changes():
    # Get files changed in this commit
    changed_files = subprocess.check_output(
        ["git", "diff", "--cached", "--name-only"]
    ).decode().strip()

    if not changed_files:
        print("No changes to review")
        return

    final_response = ""

    async for message in query(
        prompt=f"Review these changed files for bugs or issues: {changed_files}",
        options=ClaudeCodeOptions(
            allowed_tools=["Read", "Grep"]  # read-only review
        )
    ):
        if message.type == "result":
            final_response = message.result

    print("Claude's review:\n", final_response)

asyncio.run(review_changes())
```

---

## TypeScript vs Python — side by side

| | TypeScript | Python |
|---|---|---|
| Import | `import { query } from "@anthropic-ai/claude-code"` | `from claude_code_sdk import query, ClaudeCodeOptions` |
| Loop | `for await (const msg of query(...))` | `async for message in query(...)` |
| Options | `options: { allowedTools: [...] }` | `options=ClaudeCodeOptions(allowed_tools=[...])` |
| Check result | `message.type === "result"` | `message.type == "result"` |
| Run async | built-in | `asyncio.run(main())` |

The logic is identical — Python just needs `asyncio.run()` to execute async code, and uses snake_case for option names.